# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant
!pip install -q matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Authors: {metadata.author}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

In [ ]:
# Show all available record sets and their fields by @id
record_sets = metadata.recordSet if hasattr(metadata, 'recordSet') else []
if not record_sets:
    # Try the data via the distribution as fallback if recordSet field is empty
    print('No record sets found in the metadata. Trying to infer from available distributions.')
    if hasattr(metadata, 'distribution'):
        print('Distributions:')
        for d in metadata.distribution:
            print(f"  - Distribution @id: {getattr(d, '@id', None)}")
    else:
        print('No distributions found either.')
else:
    for rs in record_sets:
        print(f"RecordSet @id: {getattr(rs, '@id', None)} | name: {getattr(rs, 'name', None)}")
        if hasattr(rs, 'field'):
            print(f"  Fields (@id):")
            for fld in rs.field:
                print(f"    - {getattr(fld, '@id', None)} | name: {getattr(fld, 'name', None)}")

Since there is no `recordSet` declared in the top-level metadata but only a `distribution`, we will list the accessible record set `@id`s directly from the dataset object.

In [ ]:
# List available record set @ids from the dataset (uses internal indexing)
record_set_ids = list(dataset.record_sets.keys())
print('Available record_set @id values:')
for rid in record_set_ids:
    print(f"  - {rid}")

# For the first record set, print fields (by @id)
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"\nFields in RecordSet '{first_rs}':")
    field_objs = dataset.record_sets[first_rs]['fields'] if 'fields' in dataset.record_sets[first_rs] else []
    for fld in field_objs:
        print(f"  - {fld['@id']} (name: {fld.get('name', '')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We use record set and field `@id`s from the overview.

In [ ]:
# Extract all record sets into Pandas DataFrames using their @id
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from record set '{record_set_id}'")
    except Exception as e:
        print(f"Could not load records for record set {record_set_id}: {e}")

# Preview the columns and sample rows in the main record set
main_record_set = record_set_ids[0] if record_set_ids else None
if main_record_set and main_record_set in dataframes:
    print(f"\nColumns in main record set: {dataframes[main_record_set].columns.tolist()}")
    display(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming distributions, grouping by key attributes, and more.

In [ ]:
# Pick a numeric field for demonstration. Adjust the field name (id) based on prior output.

df = dataframes[main_record_set].copy() if main_record_set in dataframes else pd.DataFrame()
print("Available fields for numeric analysis:")
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        print(f"- {col}")

# Suppose the numeric field of interest is 'cr:coverage_percent' and the grouping is 'cr:description'.
numeric_field_id = None
group_field_id = None

for col in df.columns:
    if 'cover' in col or 'percent' in col:
        numeric_field_id = col
    if 'desc' in col:
        group_field_id = col

# Set defaults if not found
if numeric_field_id is None and len(df.columns):
    numeric_field_id = df.columns[0]
if group_field_id is None and len(df.columns) > 1:
    group_field_id = df.columns[1]

print(f"Selected numeric field: {numeric_field_id}")
print(f"Selected group field: {group_field_id}")

# Filter for values greater than a threshold (arbitrary, e.g., 10)
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (total: {len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize
    filtered_df.loc[:, f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group and aggregate (mean)
    if group_field_id and group_field_id in filtered_df:
        group_obj = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        display(group_obj.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
if numeric_field_id in df.columns:
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), kde=True, bins=30)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

# If grouping field exists, show boxplot
if numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(12,5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we've used the `mlcroissant` library to access and explore a FAIR² mass spectrometry dataset for extracellular vesicles from human mast cells. We loaded the data using Croissant schema, explored available record sets using `@id`, filtered and normalized a numeric field, grouped the data, and produced visualizations for distribution and group-wise comparison.

You can now adapt this template for more advanced analyses or specific downstream tasks relevant to your biomedical or proteomics research.